In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np                  # numerical operations and array handling
import pandas as pd                 # data loading, cleaning, and manipulation
import seaborn as sns               # statistical data visualization
import matplotlib.pyplot as plt     # plot rendering and figure control
import warnings                     # suppress non-critical warnings during EDA/modeling

warnings.filterwarnings(
    "ignore", category=FutureWarning
)

from sklearn.linear_model import LogisticRegression     # baseline classification model
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)                                                       # model evaluation metrics
from sklearn.preprocessing import StandardScaler        # feature scaling for linear models
from sklearn.pipeline import Pipeline                   # preprocessing and modeling
from sklearn.ensemble import RandomForestClassifier     # tree-based ensemble model

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Exploratory Data Analysis

In [ ]:
# load training data
train_data = pd.read_csv("/kaggle/input/titanic/train.csv")
train_data.head(5)

In [ ]:
# load test data 
test_data = pd.read_csv("/kaggle/input/titanic/test.csv")
test_data.head(5)


**Explore Data to see if any trends initially pop out**

In [ ]:
# convert to dataframe and assess shape
train_df = train_data 
train_df.shape

**Dataset has 891 rows / passengers and 12 columns**

In [ ]:
# assess information 
train_df.info()

In [ ]:
# statistical information on train data
train_df.describe()

**Above we see there are 891 passengers in this training data. The average age is 29.7 years and 38% survived.**

In [ ]:
# assess totals of missing data 
train_df.isnull().sum()

**Age is missing for 177 passengers. Cabin is missing for 687 passengers. Cabin will be unuseable for this analysis. Age can be imputed with the average age. Mode can be used for teh 2 missing values in embarked.**

In [ ]:
# visualize missing values 
sns.heatmap(train_df.isna(), cbar=False)

In [ ]:
# check columns
train_df.columns

In [ ]:
# assess survival rate
train_df['Survived'].value_counts(normalize=True)

**38.3% survived, 61.6% died**

In [ ]:
# plot survival distribution
counts = train_df['Survived'].value_counts().sort_index()

plt.bar(['Died', 'Survived'], counts, color=['red', 'green'])
plt.xlabel('Survival Status')
plt.ylabel('Passenger Count')
plt.title('Survival Distribution')
plt.show()


In [ ]:
# univariate analysis 
sns.histplot(train_df['Age'], bins=30)

In [ ]:
# plot embark
sns.countplot(x='Embarked', data=train_df)

In [ ]:
# boxplot: age by survival 
plt.figure(figsize=(6, 4))

ax = sns.boxplot(
    data=train_df,
    x='Survived',
    y='Age',
    hue='Survived',
    palette=['red', 'green'],
    dodge=False,
    legend=False
)

plt.xlabel('Survival Status')
plt.ylabel('Age')
plt.title('Age Distribution by Survival Status')
plt.show()

In [ ]:
# bivariate analysis of age and survival 
plt.figure(figsize=(8, 4))
ax = sns.histplot(
    data=train_df,
    x='Age',
    hue='Survived',
    hue_order=[0, 1],
    bins=30,
    kde=True,
    palette={0: 'red', 1: 'green'}
)

plt.xlabel('Age')
plt.ylabel('Passenger Count')
plt.title('Age Distribution by Survival')

# legend set up
legend = ax.get_legend()
mapping = {'0': 'Died', '1': 'Survived', '0.0': 'Died', '1.0': 'Survived'}

for text in legend.get_texts():
    text.set_text(mapping.get(text.get_text(), text.get_text()))

legend.set_title('Survival Status')

plt.show()


**Shows that younger people were more likely to survive**

In [ ]:
# assess gender survival 
train_df.groupby('Sex')['Survived'].mean()


In [ ]:
# plot survival by sex 
sns.barplot(x='Sex', y='Survived', data=train_df)
plt.title('Survival Rate by Sex')

**Women were more likely to survive**

In [ ]:
train_df.groupby(['Sex', 'Pclass'])['Survived'].mean().unstack()


In [ ]:
# plot gender and passenger class 
sns.barplot(
    x='Pclass',
    y='Survived',
    hue='Sex',
    data=train_df
)
plt.title('Survival Rate by Class and Sex')


**Women in all classes were more likely to survive**

In [ ]:
# assess fare vs sex
train_df.groupby('Sex')['Fare'].median()
train_df.groupby(['Sex', 'Survived'])['Fare'].median()


In [ ]:
# plot fare survival and sex 
sns.boxplot(
    x='Survived',
    y='Fare',
    hue='Sex',
    data=train_df
)
plt.yscale('log')
plt.title('Fare by Survival and Sex')


**Shows trend that fare is associated with survival especially among women**

In [ ]:
train_df[train_df['Sex'] == 'female'].groupby('Survived')['Age'].median()


In [ ]:
# age of female survivors 
sns.histplot(
    data=train_df[train_df['Sex'] == 'female'],
    x='Age',
    hue='Survived',
    bins=30,
    kde=True
)
plt.title('Age Distribution of Female Passengers by Survival')


In [ ]:
# assess sex, class and age 
sns.boxplot(
    x='Pclass',
    y='Age',
    hue='Survived',
    data=train_df[train_df['Sex'] == 'female']
)
plt.title('Age by Class and Survival (Females)')


**Shows passenger class has higher survivability and as class decreases age of survival decreases**

**This exploratory data analysis shows that survival outcomes differed significantly by sex and class. Female passengers consistently demonstrated higher survival rates across all classes. First class females had the highest survival probability. Higher fares were associated with improved survival outcomes, particularly with female passengers. This indicates socioeconomic status as a key factor. Age also played a role, with younger female passengers having higher survival likelihood especially among children.**

# Data Cleaning

In [ ]:
# drop columns that are not important and / or have to many missing values
train_df = train_df.drop(columns=['Cabin', 'Ticket', 'Name'])
train_df.columns

In [ ]:
# fix invalid entries 
train_df= train_df.replace([np.inf, -np.inf],np.nan)

In [ ]:
# replace missing age values with the grouped median values
train_df['Age'] = train_df.groupby(['Sex', 'Pclass'])['Age'] \
                          .transform(lambda x: x.fillna(x.median()))


In [ ]:
# check that imputation took place 
train_df['Age'].isnull().sum()

**Age column has no more missing values**

In [ ]:
# fill missing values in embarked wtih mode 
train_df['Embarked'].fillna(train_df['Embarked'].mode()[0], inplace=True)

In [ ]:
# check that imputation took place
train_df['Embarked'].isnull().sum()

**Embarked column has no missing values**

In [ ]:
# feature engingeering -- family features
train_df['FamilySize'] = train_df['SibSp'] + train_df['Parch'] + 1
train_df['IsAlone'] = (train_df['FamilySize'] == 1).astype(int)


In [ ]:
# clean up fare due to skewing
train_df['Fare'] = train_df['Fare'].fillna(train_df['Fare'].median())

In [ ]:
# encode categirical variables 
train_df = pd.get_dummies(
    train_df,
    columns=['Sex', 'Embarked'],
    drop_first=True
)

In [ ]:
# verify no more missing values 
train_df.isna().sum()
train_df.dtypes

In [ ]:
# feature separation and identify target
X = train_df.drop(columns=['Survived', 'PassengerId'])
y = train_df['Survived']


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=25
)


**After completing EDA, I cleaned the data by removing high-missing columns, imputing age using grouped medians informed by passenger class and sex, engineering family-based features, addressing skewed fare distributions, encoding categorical variables, and validating that the final dataset contained no missing values before modeling.**

# Modeling

In [ ]:
# standard logistic regression model 
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_val)

print("Accuracy:", accuracy_score(y_val, y_pred))
print(classification_report(y_val, y_pred))

**The logistic regression has an accuracy of 81%. This could be improved, however it validates the exploratory data analysis and cleaning steps. Next steps are to try a non-linear model.**

In [ ]:
# perform random forest classifier model 

rf = RandomForestClassifier(
    n_estimators=500,
    random_state=25
)

rf.fit(X_train, y_train)
rf_pred = rf.predict(X_val)

print("RF Accuracy:", accuracy_score(y_val, rf_pred))
print(classification_report(y_val, rf_pred))

**The random forest classifier is slightly more accurate.**

In [ ]:
# identify feature importance

feature_importance = pd.Series(
    rf.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

feature_importance


In [ ]:
# passenger ID should not be important - we need to remove this
#X_train = X_train.drop(columns=['PassengerId'])
#X_val = X_val.drop(columns=['PassengerId'])

In [ ]:
# rerun the random forest classifier 
rf = RandomForestClassifier(
    n_estimators=500,
    random_state=25
)

rf.fit(X_train, y_train)
rf_pred = rf.predict(X_val)

print("RF Accuracy:", accuracy_score(y_val, rf_pred))
print(classification_report(y_val, rf_pred))

**Removing passenger ID decreased model accurac. This was expected. **

In [ ]:
# recheck feature importances 
feature_importance = pd.Series(
    rf.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

feature_importance


**Feature importance analysis from the Random Forest model shows that fare, passenger sex, and age are the most influential predictors of survival, followed by passenger class. Family structure and embarkation location contribute modestly, indicating that socioeconomic status and demographic factors play the primary role in survival outcomes.**

# Prepare the Test Data

In [ ]:
# test data
test_data.head(5)

In [ ]:
# create a working copy of the Kaggle test set
test_df = test_data.copy()

# keep PassengerId for the submission file, but do not use it as a feature
test_passenger_id = test_df['PassengerId']


In [ ]:
# quick missingness check
test_df.isna().sum()


In [ ]:
# apply the same cleaning / preprocessing steps used for training

# drop columns not used for modeling
test_df = test_df.drop(columns=['Cabin', 'Ticket', 'Name'])

# replace infinite values
test_df = test_df.replace([np.inf, -np.inf], np.nan)

# impute Age using grouped medians (Sex x Pclass)
test_df['Age'] = test_df.groupby(['Sex', 'Pclass'])['Age'] \
                        .transform(lambda x: x.fillna(x.median()))

# fill Embarked with mode (very few missing)
test_df['Embarked'].fillna(train_data['Embarked'].mode()[0], inplace=True)

# fare has missing values in test set
test_df['Fare'].fillna(train_data['Fare'].median(), inplace=True)

# feature engineering
test_df['FamilySize'] = test_df['SibSp'] + test_df['Parch'] + 1
test_df['IsAlone'] = (test_df['FamilySize'] == 1).astype(int)

# one-hot encode categorical variables (match training)
test_df = pd.get_dummies(
    test_df,
    columns=['Sex', 'Embarked'],
    drop_first=True
)

# align test columns to training feature columns
test_df = test_df.reindex(columns=X.columns, fill_value=0)

test_df.head()


In [ ]:
# fill fare with median for 1 missing value
test_df['Fare'].fillna(test_df['Fare'].median(), inplace=True)

In [ ]:
# fit the final model on the full cleaned training dataset
pipeline.fit(X, y)


In [ ]:
# generate predictions for Kaggle submission
test_predictions = pipeline.predict(test_df)

submission = pd.DataFrame({
    'PassengerId': test_passenger_id,
    'Survived': test_predictions
})

submission.to_csv('submission.csv', index=False)
submission.head()


In [ ]:
# check submission csv
submission.head()
submission.shape

In [ ]:
# save submission
submission.to_csv('submission.csv', index=False)
